# Part 2 — Notebook 08: Scripts — Tự động hóa

**Vibe Coding Guide cho người KHÔNG biết lập trình**

---

> Scripts = "nút bấm" — bấm 1 nút, máy tự chạy hàng chục bước kiểm tra.

## Scripts là gì?

Hãy nghĩ về **nhà máy sản xuất**.

Cách 1 (thủ công): Công nhân tự kiểm tra từng sản phẩm bằng tay.
- Mỗi người kiểm tra khác nhau
- Có người quên bước, có người làm sai thứ tự
- Tốn thời gian, không nhất quán

Cách 2 (tự động): Bấm nút, máy kiểm tra tự động.
- Luôn kiểm tra đúng các bước
- Không bỏ sót
- Nhanh và nhất quán

**Scripts = nút bấm tự động.** Thay vì AI gõ 10 lệnh phức tạp, AI chỉ cần chạy 1 script.

## Tại sao BẮT BUỘC dùng scripts?

Bài học thực tế (đã xảy ra trong dự án):

| Tình huống | Chạy lệnh trực tiếp | Dùng script |
|-----------|---------------------|-------------|
| Quên 1 bước kiểm tra | Lỗi lọt vào code | Script kiểm tra đủ bước |
| Gõ sai lệnh | Hỏng dữ liệu/môi trường | Script đã được test |
| Mỗi lần chạy khác nhau | Kết quả không nhất quán | Luôn chạy giống nhau |
| AI "sáng tạo" lệnh mới | Có thể nguy hiểm | Chỉ chạy lệnh đã duyệt |

**Quy tắc trong CLAUDE.md:**
```
KHÔNG BAO GIỜ chạy lệnh trực tiếp.
LUÔN LUÔN dùng scripts.
```

## Các scripts trong Kit

Kit cung cấp sẵn các scripts thiết yếu:

```
.claude/scripts/
├── pre-commit-check.sh     # Kiểm tra TRƯỚC khi commit
├── commit-msg-check.sh     # Kiểm tra commit message
├── test-local.sh           # Chạy test trước khi push (dự án tự thêm)
├── check-ci.sh             # Theo dõi CI trên GitHub (dự án tự thêm)
├── render-diagrams.sh      # Render sơ đồ (dự án tự thêm)
└── capture-screenshots.ts  # Chụp ảnh giao diện (dự án tự thêm)
```

Hãy tìm hiểu từng script.

### 1. pre-commit-check.sh — Gác cổng trước khi commit

**Ví von:** Như bảo vệ kiểm tra trước khi vào tòa nhà.

Khi AI chuẩn bị "lưu" code (commit), script này tự động chạy và kiểm tra:

- Commit message có đúng format không? (ví dụ: `feat(core): add login`)
- Có file nhạy cảm nào bị thêm vào không? (mật khẩu, API key...)
- Có TODO/FIXME nào bị quên không?
- Code có lỗi cú pháp cơ bản không?

Nếu **bất kỳ kiểm tra nào fail** → commit bị **chặn lại**. AI phải sửa rồi thử lại.

In [ ]:
# Mô phỏng pre-commit check
echo "=== Pre-Commit Check ==="
echo ""
echo "[1/4] Kiểm tra commit message format..."
echo "      'feat(core): add user login' -> OK"
echo ""
echo "[2/4] Kiểm tra file nhạy cảm..."
echo "      Không có .env, credentials.json -> OK"
echo ""
echo "[3/4] Kiểm tra TODO/FIXME..."
echo "      Không có TODO mới -> OK"
echo ""
echo "[4/4] Kiểm tra cú pháp..."
echo "      Compile thành công -> OK"
echo ""
echo "--- KẾT QUẢ: ALL PASSED --- Commit được phép!"

In [ ]:
# Mô phỏng khi pre-commit FAIL
echo "=== Pre-Commit Check ==="
echo ""
echo "[1/4] Kiểm tra commit message format..."
echo "      'fix bug' -> FAIL!"
echo "      Commit message phai theo format: type(scope): description"
echo "      Vi du: 'fix(auth): resolve login timeout'"
echo ""
echo "--- KẾT QUẢ: BLOCKED --- Commit bị chặn! AI phải sửa commit message."

### 2. commit-msg-check.sh — Kiểm tra commit message

**Ví von:** Như kiểm tra định dạng đơn xin phép — sai format thì trả lại.

Commit message = lời mô tả ngắn về thay đổi. Cần theo format:

```
type(scope): mô tả ngắn

Ví dụ:
feat(auth): add Google login
fix(payment): resolve timeout error
docs(readme): update installation guide
```

| Type | Ý nghĩa |
|------|----------|
| `feat` | Tính năng mới |
| `fix` | Sửa lỗi |
| `docs` | Thay đổi tài liệu |
| `refactor` | Cải thiện code (không đổi chức năng) |
| `test` | Thêm/sửa test |

Script kiểm tra: đúng format, dưới 50 ký tự, không có từ cấm.

### 3. test-local.sh — Chạy test trước khi push

**Ví von:** Như kiểm tra hàng trước khi gửi cho khách.

Script này:
- Tự phát hiện phần code nào thay đổi
- Chạy test CHỈ cho phần thay đổi (nhanh hơn test tất cả)
- Có chế độ nhanh (quick): chỉ compile/lint, không chạy full test

```bash
# Chạy test đầy đủ
./scripts/test-local.sh

# Chạy nhanh (chỉ compile + lint)
./scripts/test-local.sh --quick
```

Kết quả: PASS hoặc FAIL + chi tiết lỗi.

In [ ]:
# Mô phỏng test-local.sh
echo "=== Test Local ==="
echo ""
echo "Detecting changes..."
echo "  Modified: src/auth/LoginService.java"
echo "  Modified: src/auth/LoginController.java"
echo ""
echo "Running tests for: auth module"
echo "  LoginServiceTest......... PASS (0.3s)"
echo "  LoginControllerTest...... PASS (0.8s)"
echo "  LoginIntegrationTest..... PASS (2.1s)"
echo ""
echo "Results: 3 tests, 3 passed, 0 failed"
echo "Time: 3.2 seconds"
echo ""
echo "--- ALL TESTS PASSED --- An toàn để push!"

### 4. check-ci.sh — Theo dõi CI trên GitHub

**Ví von:** Như theo dõi đơn hàng online — chờ xem "đã giao" hay "bị lỗi".

Sau khi AI push code lên GitHub, hệ thống CI (Continuous Integration) tự động chạy test. Script này:
- Theo dõi tiến trình CI
- Chờ đến khi xong
- Báo kết quả: PASS hoặc FAIL

```bash
# Chờ CI hoàn thành
./scripts/check-ci.sh

# Chỉ xem trạng thái nhanh
./scripts/check-ci.sh --status
```

In [ ]:
# Mô phỏng check-ci.sh
echo "=== CI Status ==="
echo ""
echo "Branch: feature/KC-301-class-management"
echo "Commit: abc1234 'feat(core): add class CRUD endpoints'"
echo ""
echo "Workflows:"
echo "  [OK] Build Java.............. 45s"
echo "  [OK] Unit Tests.............. 1m20s"
echo "  [OK] Integration Tests....... 2m15s"
echo "  [OK] Lint & Checkstyle....... 30s"
echo "  [OK] Build Frontend.......... 1m05s"
echo ""
echo "--- ALL CI CHECKS PASSED ---"
echo "PR sẵn sàng để review và merge!"

### 5. render-diagrams.sh — Render sơ đồ

**Ví von:** Như máy in bản vẽ — bạn viết mô tả, máy in ra hình.

Trong dự án, sơ đồ kỹ thuật được viết dưới dạng văn bản (PlantUML hoặc Mermaid). Script này chuyển văn bản thành hình ảnh PNG:

```
Văn bản (input):          Hình ảnh (output):
@startuml                  ┌──────────┐
User -> Server : login     │  User    │──login──>│ Server │
Server -> DB : query       │          │          │        │──query──>│ DB │
@enduml                    └──────────┘
```

### 6. capture-screenshots.ts — Chụp ảnh giao diện

**Ví von:** Như chụp ảnh "trước/sau" khi sửa nhà.

Script này:
- Tự mở website trên trình duyệt ẩn
- Chụp ảnh mỗi trang
- Lưu vào folder có nhãn (before/after)
- Dùng để so sánh giao diện trước và sau khi sửa

```bash
# Chụp ảnh trước khi sửa
npx ts-node scripts/capture-screenshots.ts --label before-fix

# Sửa code...

# Chụp ảnh sau khi sửa
npx ts-node scripts/capture-screenshots.ts --label after-fix
```

Kết quả: 2 bộ ảnh để so sánh trực quan.

## Git Hooks — Scripts tự động chạy

Một số scripts được cài đặt như **hooks** — nghĩa là tự động chạy mà không cần bạn bảo.

| Hook | Khi nào chạy | Script |
|------|-------------|--------|
| `pre-commit` | Trước mỗi commit | pre-commit-check.sh |
| `commit-msg` | Kiểm tra commit message | commit-msg-check.sh |

**Cách hoạt động:**
1. AI chạy `git commit`
2. Git tự động gọi `pre-commit` hook
3. Hook chạy `pre-commit-check.sh`
4. Nếu PASS → commit thành công
5. Nếu FAIL → commit bị chặn, AI phải sửa

Bạn không cần làm gì — hooks tự bảo vệ code của bạn.

In [ ]:
# Mô phỏng quy trình hook
echo "=== Git Commit Flow ==="
echo ""
echo "AI chạy: git commit -m 'feat(auth): add login page'"
echo ""
echo "  [Hook 1] pre-commit-check.sh"
echo "    - Kiểm tra file nhạy cảm... OK"
echo "    - Kiểm tra TODO/FIXME....... OK"
echo "    - Compile code.............. OK"
echo "  -> PASSED"
echo ""
echo "  [Hook 2] commit-msg-check.sh"
echo "    - Format đúng (type(scope): desc)... OK"
echo "    - Dưới 50 ký tự.................... OK"
echo "    - Không có từ cấm.................. OK"
echo "  -> PASSED"
echo ""
echo "COMMIT THÀNH CÔNG!"

## So sánh: Lệnh trực tiếp vs Script

### Ví dụ thực tế: Kiểm tra trước khi push code

**Cách SAI (lệnh trực tiếp):**
```bash
# AI tự gõ 5 lệnh riêng lẻ
mvn compile                    # Có thể quên
mvn test                       # Có thể quên scope
mvn checkstyle:check           # Có thể quên
npm run lint                   # Có thể gõ sai
npm run test                   # Có thể quên
```

**Cách ĐÚNG (dùng script):**
```bash
# 1 lệnh, tất cả kiểm tra
./scripts/test-local.sh
```

Script xử lý: detect thay đổi, chạy đúng test, báo kết quả rõ ràng.

## Bạn (non-dev) cần biết gì?

1. **Scripts = bảo hiểm cho dự án** — ngăn AI commit code lỗi
2. **Bạn không cần chạy scripts** — AI tự chạy, hooks tự kích hoạt
3. **Khi AI báo "test failed"** — nghĩa là script đã bắt được lỗi (tốt!)
4. **Thêm script mới được** — nhờ AI tạo script cho nhu cầu riêng

Ví dụ bạn có thể nhờ AI:
- "Tạo script kiểm tra broken links trên website"
- "Tạo script backup database mỗi ngày"
- "Tạo script deploy lên server"

In [ ]:
# Liệt kê scripts có sẵn trong Kit
echo "=== Scripts trong Starter Kit ==="
echo ""
echo "Tên script                  | Mục đích"
echo "----------------------------|---------------------------------"
echo "pre-commit-check.sh         | Kiểm tra trước khi commit"
echo "commit-msg-check.sh         | Kiểm tra format commit message"
echo "test-local.sh (*)           | Chạy test local trước khi push"
echo "check-ci.sh (*)             | Theo dõi CI trên GitHub"
echo "render-diagrams.sh (*)      | Render PlantUML/Mermaid -> PNG"
echo "capture-screenshots.ts (*)  | Chụp ảnh giao diện"
echo ""
echo "(*) = Script mà dự án tự thêm dựa trên nhu cầu"
echo "    Kit cung cấp framework, dự án customize thêm"

---

**Tóm tắt Notebook 08:**

- Scripts = "nút bấm" tự động hóa, thay thế lệnh thủ công
- AI BẮT BUỘC dùng scripts, KHÔNG chạy lệnh trực tiếp
- 2 scripts cốt lõi: pre-commit-check (gác cổng) + commit-msg-check (kiểm tra format)
- Git hooks = scripts tự động chạy, không cần nhắc
- Scripts = bảo hiểm cho dự án, ngăn code lỗi lọt vào